# SupportRoute — Agent Experience Hackathon: Strands Decider

**AI-powered customer support routing with Strands Decider 2B and Amazon Bedrock**

This notebook walks through the essential code behind SupportRoute:
- The reusable `#policy` decision pattern
- The Strands tools
- The full agent
- Live classification demos
- Benchmark results

Built with the **AWS User Group Madurai** Strands Decider hackathon kit.

---
**Setup:** API keys and model settings are read from `.env` (see `.env.example`).
No keys appear in this notebook.

## 1. Environment setup

Install dependencies and load settings from `.env`.

In [ ]:
# Install the kit's dependencies (already done if you used ./kit setup)
# %pip install -q strands-agents strands-agents-tools python-dotenv

import os
from pathlib import Path
from dotenv import load_dotenv

# Load .env from the repo root — contains DECIDER_API_KEY, LLM_PROVIDER, etc.
# Keys are never printed or stored here.
env_path = Path("../.env")
if env_path.exists():
    load_dotenv(env_path)
    print("Loaded .env")
else:
    print("No .env found — set DECIDER_MODE, DECIDER_URL and DECIDER_API_KEY manually")

print(f"LLM provider : {os.getenv('LLM_PROVIDER', 'bedrock')}")
print(f"Decider mode : {os.getenv('DECIDER_MODE', 'local')}")
print(f"Bedrock model: {os.getenv('BEDROCK_MODEL_ID', 'us.amazon.nova-pro-v1:0')}")

## 2. The reusable decision pattern (`#policy`)

This is the core of SupportRoute. One `Decider.ask()` call with two typed questions:
1. **category** (choice) — which support team should handle this?
2. **urgent** (noul) — is the customer expressing urgency or frustration?

This pattern lives in `agent/decision.py` and is designed to be dropped into any
support agent. Change the category options to match your business.

In [ ]:
from decider_kit import Decider

# ── Pattern metadata ─────────────────────────────────────────────────────────
PATTERN_TAG = "#policy"
LOW_CONFIDENCE_THRESHOLD = 0.65

# ── Typed questions ───────────────────────────────────────────────────────────
QUESTIONS = {
    "category": Decider.choice(
        "Which support category best matches this customer message?",
        {
            "billing":        "Customer is asking about an invoice, charge, payment, subscription cost, plan upgrade or downgrade, refund, or any question about what they are being billed for — even if they mention urgency or deadlines around the payment",
            "technical":      "Customer reports a bug, crash, error message, feature not working, data not syncing, API failure, or performance problem — even if the impact is severe or production-affecting",
            "account-access": "Customer cannot log in, needs a password reset, has 2FA problems, or their account is locked or blocked",
            "general":        "Customer has a general product question, how-to request, feature request, feedback, or the message is unclear and does not fit another category",
            "escalation":     "Customer is expressing anger, frustration, threatening to cancel or leave, demanding a manager, or saying the service is unacceptable — the primary signal is emotional tone, not the topic itself",
        },
    ),
    "urgent": Decider.noul(
        "Is the customer expressing urgency or strong frustration?",
        {
            "true":  "Customer uses words like urgent, immediately, now, furious, unacceptable, broken production, losing business, deadline today, or makes an explicit demand for immediate action",
            "false": "Customer's tone is neutral, patient, or politely asking for help without pressure",
        },
    ),
}

print(f"Pattern tag : {PATTERN_TAG}")
print(f"Questions   : {list(QUESTIONS.keys())}")
print(f"Categories  : {list(QUESTIONS['category']['options'].keys())}")

## 3. The SupportRouter — wiring the decider

`SupportRouter.classify(message)` is the single call you make from the agent.
It returns the category, confidence, urgency flag, and per-category probabilities.

In [ ]:
class SupportRouter:
    """
    Wires Strands Decider 2B into a support routing agent.

    How to reuse:
        router = SupportRouter()
        result = router.classify(customer_message)
        if result["low_confidence"]:
            ask_for_clarification()
        else:
            route_to(result["category"], urgent=result["urgent"])
    """

    def __init__(self):
        self._decider = Decider()  # reads DECIDER_MODE / DECIDER_URL / DECIDER_API_KEY from .env

    def classify(self, message: str) -> dict:
        """
        Classify a customer support message.

        Returns:
            category       — one of: billing, technical, account-access, general, escalation
            confidence     — probability of the top category (0..1)
            urgent         — True if the customer seems urgent or frustrated
            low_confidence — True if confidence < LOW_CONFIDENCE_THRESHOLD (0.65)
            decider_ms     — time the decider took in milliseconds
            probabilities  — full probability dict for all five categories
        """
        state = f"Customer support message:\n{message}"
        answers = self._decider.ask(state, QUESTIONS)

        category = answers["category"].label
        confidence = answers["category"].confidence
        urgent = answers["urgent"].yes
        probabilities = {k: round(v, 3) for k, v in answers["category"].probabilities.items()}

        return {
            "category":       category,
            "confidence":     round(confidence, 3),
            "urgent":         urgent,
            "low_confidence": confidence < LOW_CONFIDENCE_THRESHOLD,
            "decider_ms":     round(self._decider.last_latency_ms),
            "probabilities":  probabilities,
        }

print("SupportRouter defined")

## 4. Team routing data

Each category maps to a named support team, ETA, and description.
In the full agent this is loaded from `agent/data/teams.json`.

In [ ]:
import json, random, string

TEAMS = {
    "billing": {
        "team": "Billing & Payments",
        "eta": "within 4 hours",
        "description": "Handles invoices, charges, payments, subscriptions, and refunds"
    },
    "technical": {
        "team": "Technical Support",
        "eta": "within 2 hours",
        "description": "Handles bugs, crashes, errors, app issues, and performance problems"
    },
    "account-access": {
        "team": "Account Security",
        "eta": "within 1 hour",
        "description": "Handles login issues, password resets, 2FA problems, and locked accounts"
    },
    "general": {
        "team": "Customer Success",
        "eta": "within 24 hours",
        "description": "Handles product questions, how-to requests, feedback, and general inquiries"
    },
    "escalation": {
        "team": "Senior Support",
        "eta": "within 30 minutes",
        "description": "Handles urgent escalations, frustrated customers, and high-priority issues"
    },
}

def random_ticket_id():
    suffix = "".join(random.choices(string.ascii_uppercase + string.digits, k=6))
    return f"TKT-{suffix}"

print("Teams defined:", list(TEAMS.keys()))

## 5. The two Strands tools

- `classify_request` — calls the decider, returns category + urgency
- `create_ticket` — assigns a team, priority, and reference number

In [ ]:
from strands import tool

_router = SupportRouter()
_decision_log = []  # populated per turn so the agent can return decider results


@tool
def classify_request(message: str) -> dict:
    """
    Classify a customer support message using Strands Decider 2B.

    Returns category, confidence, urgent flag, low_confidence flag,
    decider latency in ms, and per-category probabilities.
    """
    result = _router.classify(message)
    _decision_log.append({
        "probabilities": result["probabilities"],
        "verdict":       result["category"],
        "decider_ms":    result["decider_ms"],
    })
    return result


@tool
def create_ticket(category: str, urgent: bool, summary: str) -> dict:
    """
    Create a support ticket and return team, ETA, priority, and ticket ID.

    Call this after classify_request when low_confidence is False.
    """
    if urgent and category != "escalation":
        priority = "urgent"
    elif category == "escalation":
        priority = "critical"
    else:
        priority = "normal"

    team_info = TEAMS.get(category, TEAMS["general"])
    return {
        "ticket_id": random_ticket_id(),
        "team":      team_info["team"],
        "eta":       team_info["eta"],
        "priority":  priority,
        "category":  category,
        "summary":   summary,
    }


print("Tools defined: classify_request, create_ticket")

## 6. Direct decider demo — no agent, just the classifier

Call `SupportRouter.classify()` directly to see how the decider reads each message.

In [ ]:
demo_messages = [
    "I was charged twice for my subscription this month.",
    "The app crashes every time I try to open the dashboard.",
    "I can't log in — my account is locked after failed attempts.",
    "This is absolutely unacceptable! I want to speak to a manager immediately!",
    "Something feels off, not sure if it's billing or my account.",  # low-confidence case
]

router = SupportRouter()

print(f"{'Message':<55} {'Category':<16} {'Conf':>5}  {'Urgent':>6}  {'Low?':>5}  {'ms':>6}")
print("-" * 100)
for msg in demo_messages:
    r = router.classify(msg)
    low = "YES" if r["low_confidence"] else "-"
    line = f"{msg[:54]:<55} {r['category']:<16} {r['confidence']:>5.2f}  {str(r['urgent']):>6}  {low:>5}  {r['decider_ms']:>6} ms"
    print(line)


## 7. Full probability breakdown

The decider returns a calibrated probability for every category in one pass.
This is what powers the confidence threshold and low-confidence fallback.

In [ ]:
test_message = "I was charged twice for my subscription this month."
result = router.classify(test_message)

print(f"Message  : {test_message}")
print(f"Verdict  : {result['category']} (confidence {result['confidence']:.2f})")
print(f"Urgent   : {result['urgent']}")
print(f"Latency  : {result['decider_ms']} ms")
print()
print("Per-category probabilities:")
for cat, prob in sorted(result["probabilities"].items(), key=lambda x: -x[1]):
    bar = "#" * int(prob * 40)
    print(f"  {cat:<16} {prob:.3f}  {bar}")

## 8. The full agent — HackathonAgent

The agent uses Amazon Bedrock (or the user group cloud during development)
for the conversation, with the two tools above. The decider runs before the
LLM composes any reply.

In [ ]:
import time
from strands import Agent
from decider_kit.llm import agent_model

SYSTEM_PROMPT = """\
You are SupportRoute, a customer support routing assistant.

Your job:
1. Call classify_request with the customer's message.
2. If low_confidence is True: ask the customer to describe their issue in more detail.
3. If low_confidence is False: call create_ticket with the category, urgent flag,
   and a short one-line summary of the issue (10 words max).
4. Reply with: the team name, ticket reference, ETA, and a high-priority note if urgent.

Keep your reply friendly, concise, and reassuring. One short paragraph.
"""


class SupportRouteAgent:
    """Full SupportRoute agent — Strands + Decider + Bedrock."""

    def __init__(self, model_id=None):
        self._model = agent_model(model_id=model_id)
        self._agent = Agent(
            model=self._model,
            system_prompt=SYSTEM_PROMPT,
            tools=[classify_request, create_ticket],
        )

    def chat(self, message: str) -> dict:
        t0 = time.time()
        _decision_log.clear()

        reply = str(self._agent(message))
        latency_ms = round((time.time() - t0) * 1000)
        model_id = getattr(self._model, "model_id", None) or getattr(self._model, "_model_id", "unknown")

        return {
            "reply":      reply,
            "decisions":  list(_decision_log),
            "model_id":   str(model_id),
            "latency_ms": latency_ms,
        }


print("SupportRouteAgent defined")
print("Call: agent = SupportRouteAgent(); result = agent.chat('your message here')")

## 9. Run a full agent conversation

This cell sends a real message through the agent — LLM + decider + tools.

In [ ]:
agent = SupportRouteAgent()

test_cases = [
    "I was charged twice for my subscription this month.",
    "This is absolutely unacceptable! I want a manager NOW!",
]

for message in test_cases:
    print(f"Customer: {message}")
    result = agent.chat(message)
    print(f"Agent   : {result['reply']}")
    if result["decisions"]:
        d = result["decisions"][0]
        print(f"Decider : verdict={d['verdict']}  latency={d['decider_ms']} ms")
        print(f"Probs   : {d['probabilities']}")
    print(f"Total   : {result['latency_ms']} ms")
    print("-" * 70)

## 10. Benchmark results

These numbers come from `./kit bench --split test` against 16 held-out eval cases.
The decider ran on the AWS User Group Madurai hosted API (CPU instances).

Run `./kit bench` yourself to reproduce.

In [ ]:
import json
from pathlib import Path

results_path = Path("../submission/bench/results.json")
if results_path.exists():
    with open(results_path) as f:
        results = json.load(f)
    summary = results["summary"]
    dec = summary["decider"]
    llm = summary["llm"]

    print(f"Decider model : {results['decider_model']} ({results['decider_mode']})")
    print(f"LLM model     : {results['llm_model']}")
    print(f"Cases         : {results['cases']} held-out ({results['split']} split)")
    print()
    print(f"{'Metric':<35} {'Decider':>12} {'LLM':>12}")
    print("-" * 62)
    print(f"{'Accuracy':<35} {dec['accuracy']*100:>11.1f}% {llm['accuracy']*100:>11.1f}%")
    print(f"{'Latency p50 (ms)':<35} {dec['latency_p50_ms']:>12.0f} {llm['latency_p50_ms']:>12.0f}")
    print(f"{'Latency p95 (ms)':<35} {dec['latency_p95_ms']:>12.0f} {llm['latency_p95_ms']:>12.0f}")
    print(f"{'Cost per 1k decisions (USD)':<35} {dec['usd_per_1k_decisions']:>12.4f} {llm['usd_per_1k_decisions']:>12.4f}")
    print(f"{'Confident share (>= 0.7)':<35} {dec['confident_share']*100:>11.0f}% {llm['confident_share']*100:>11.0f}%")
    print(f"{'Accuracy when confident':<35} {dec['accuracy_when_confident']*100:>11.1f}% {llm['accuracy_when_confident']*100:>11.1f}%")
    print()
    print(f"Speedup (p50): {summary['speedup_p50']:.1f}x faster than the LLM")
else:
    print("results.json not found — run ./kit bench --split test first")

## 11. Visualise benchmark results

In [ ]:
try:
    import matplotlib.pyplot as plt
    import numpy as np

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    fig.suptitle("SupportRoute — Strands Decider 2B vs LLM", fontsize=13)

    labels = ["Strands Decider 2B", "LLM (qwen3.5-4b)"]
    colors = ["#FF9900", "#232F3E"]  # AWS orange and dark

    # Accuracy
    axes[0].bar(labels, [dec['accuracy']*100, llm['accuracy']*100], color=colors)
    axes[0].set_title("Accuracy (%)"); axes[0].set_ylim(0, 100)
    for i, v in enumerate([dec['accuracy']*100, llm['accuracy']*100]):
        axes[0].text(i, v + 1, f"{v:.0f}%", ha='center', fontweight='bold')

    # Latency p50
    axes[1].bar(labels, [dec['latency_p50_ms'], llm['latency_p50_ms']], color=colors)
    axes[1].set_title("Latency p50 (ms)"); axes[1].set_ylabel("ms")
    for i, v in enumerate([dec['latency_p50_ms'], llm['latency_p50_ms']]):
        axes[1].text(i, v + 100, f"{v:.0f} ms", ha='center', fontweight='bold')

    # Cost per 1k
    costs = [dec['usd_per_1k_decisions'], llm['usd_per_1k_decisions']]
    axes[2].bar(labels, costs, color=colors)
    axes[2].set_title("Cost per 1,000 decisions (USD)")
    for i, v in enumerate(costs):
        axes[2].text(i, v + 0.001, f"${v:.4f}", ha='center', fontweight='bold')

    plt.tight_layout()
    plt.savefig("../submission/bench/notebook_chart.png", dpi=120, bbox_inches='tight')
    plt.show()
    print("Chart saved to submission/bench/notebook_chart.png")

except ImportError:
    print("matplotlib not installed — pip install matplotlib")
except NameError:
    print("Run cell 10 first to load results.json")

## 12. Low-confidence fallback demo

When the decider's confidence is below 0.65, the agent asks for clarification
instead of misrouting. This cell shows the threshold in action.

In [ ]:
edge_cases = [
    "Something feels off, not sure if it's billing or my account.",
    "I need help, something is wrong but I'm not sure where to start.",
    "I think there might be an issue with my subscription but I'm not entirely sure.",
]

router2 = SupportRouter()
print(f"Low-confidence threshold: {LOW_CONFIDENCE_THRESHOLD}\n")

for msg in edge_cases:
    r = router2.classify(msg)
    action = "ASK FOR CLARIFICATION" if r["low_confidence"] else f"ROUTE to {TEAMS[r['category']]['team']}"
    print(f"Message   : {msg}")
    print(f"Category  : {r['category']} (conf={r['confidence']:.2f})")
    print(f"Action    : {action}")
    print()

## 13. Eval cases — the labelled dataset

28 labelled cases drive the benchmark: 16 held-out test cases, 12 tune cases.
Each case's `state` uses the same text layout the decider sees at run time.

In [ ]:
import json
from pathlib import Path

cases_path = Path("../agent/eval_cases.jsonl")
cases = [json.loads(l) for l in cases_path.read_text().splitlines() if l.strip()]

test_cases  = [c for c in cases if c.get("split") == "test"]
tune_cases  = [c for c in cases if c.get("split") == "tune"]

print(f"Total cases : {len(cases)}")
print(f"Test split  : {len(test_cases)}  (reported numbers)")
print(f"Tune split  : {len(tune_cases)}  (used for question tuning only)")
print()
print("Sample test cases:")
for c in test_cases[:4]:
    msg = c['state'].replace('Customer support message:\n', '')
    print(f"  expected={c['expected']}  |  {msg[:65]}")

---

## Summary

| Component | File |
|---|---|
| Reusable decision pattern | `agent/decision.py` |
| Strands tools | `agent/tools.py` |
| Full agent | `agent/agent.py` |
| Eval cases | `agent/eval_cases.jsonl` |
| Team data | `agent/data/teams.json` |
| Benchmark results | `submission/bench/results.json` |

**Run the full agent:** `./kit run`  
**Run the web demo:** `./kit demo`  
**Run the benchmark:** `./kit bench --split test`  
**Deploy to AWS:** `./kit deploy`

---

Built with the **AWS User Group Madurai** Strands Decider hackathon kit.  
Strands Decider: https://github.com/strands-labs/strands-decider (Apache-2.0)  
Repo: https://github.com/sivasakthivelg/support-route-agent